In [ ]:
import jax
from jax import lax
from jax import Array
import jax.numpy as jnp
from jax.scipy.special import xlogy
jax.config.update('jax_enable_x64', True)
import numpy as np
from numpy.linalg import matrix_power
from scipy.stats import entropy
from itertools import product as iproduct
from scipy.linalg import expm
from time import perf_counter
from functools import partial

import os, sys, pathlib
os.environ["OPENBLAS_NUM_THREADS"] = "1"
_project_root = pathlib.Path.cwd().resolve()
sys.path.insert(0, str(_project_root.parents[0]))
sys.path.insert(0, str(_project_root.parents[1]))
from physics.markov import H, Hv, KL, KLv, get_traj, pmmc
import physics.glauber as ising


In [ ]:
@jax.jit
def apply_glauber_old_v2(p: Array, graph: GraphT, beta, h, J=1.0):
  rows, dE_J, dE_h_sign = graph
  N = rows.shape[0]
  dE = J * dE_J + h * dE_h_sign

  # flip probability
  # off = jnp.exp(-beta * jnp.maximum(dE, 0.0)) / N
  flip_prob = jax.nn.sigmoid(-beta * dE) / N

  # prob of remaining in original state
  diag = 1.0 - flip_prob.sum(axis=0)
  p_next = diag * p
  # this is equivalent to the for loop
  idk = flip_prob * p[None, :]
  incoming = jnp.take_along_axis(idk, rows, axis=1).sum(axis=0)
  return incoming + p_next

In [ ]:
# ── Periodic boundaries ──────────────────────────────────
# def neighbors(i, L):
#     r, c = divmod(i, L)
#     return [((r-1)%L)*L + c, ((r+1)%L)*L + c,
#             r*L + (c-1)%L,   r*L + (c+1)%L]

# ── Empty boundaries ──────────────────────────────────
def neighbors(i: int, L: int) -> list[int]:
  r, c = divmod(i, L)
  nbrs = []
  if r > 0:       nbrs.append((r-1)*L + c)   # up
  if r < L-1:     nbrs.append((r+1)*L + c)   # down
  if c > 0:       nbrs.append(r*L + (c-1))   # left
  if c < L-1:     nbrs.append(r*L + (c+1))   # right
  return nbrs

def compute_site_neighbors(L: int):
  return [neighbors(i, L) for i in range(L * L)]


In [ ]:

def generate_glauber_matrix_sparse(shape=(2,2), J=1.0, h=0.1, beta=1.0, gamma=1.0):
  import scipy.sparse as sp
  N = shape[0] * shape[1]
  nstates = 1 << N
  site_neighbors = compute_site_neighbors(Lx)
  masks = [1 << (N - 1 - site) for site in range(N)]

  # Build transition matrix
  rows = []
  cols = []
  data = []

  # index here is column
  for σ in range(nstates):
    # energy for state σ
    stay_prob = 0.0

    # flip each site
    for site, mask in enumerate(masks):
      s_i = 1 if σ & mask else -1
      nb_sum = 0
      for j in site_neighbors[site]:
        nb_sum += 1 if σ & masks[j] else -1

      dE = 2 * s_i * (J * nb_sum + h)
      # integer value of η is the row of the matrix
      η = σ ^ mask
      # Metropolis acceptance probability / N
      # q(σ,eta)=1/N
      prob = gamma * np.exp(-0.5 * beta * dE)

      rows.append(η)
      cols.append(σ)
      data.append(prob)
      stay_prob -= prob
    
    rows.append(σ)
    cols.append(σ)
    data.append(stay_prob)

  return sp.coo_matrix((data, (rows, cols)), shape=(nstates, nstates)).tocsr()


In [ ]:
from scipy.sparse.linalg import expm_multiply, expm

K_sp1 = generate_glauber_matrix_sparse(shape, J, h, betas_jax[0], gammas_jax[0])
K_sp2 = generate_glauber_matrix_sparse(shape, J, h, betas_jax[1], gammas_jax[1])
K_sp = K_sp1 + K_sp2


A = K_sp * t
pt = expm_multiply(A, p0)
plt.bar(range(n_states), pt)

out = np.array(apply_K(p0, graph, J, h, betas_jax, gammas_jax))
print(np.max(np.abs((K @ p0) - out)))

In [ ]:
# ── Build K^ν — every spin coupled to every reservoir ──
def build_K(nu, L, gamma, beta, J, h):
  N = L * L
  n_states = 1 << N
  # ── States ───────────────────────────────────
  states    = [np.array(s, dtype=np.int8) for s in iproduct([-1,1], repeat=N)]
  state_idx = {tuple(s): k for k, s in enumerate(states)}

  def dE_flip(sigma, i):
    return 2.0 * sigma[i] * (J * sum(sigma[j] for j in neighbors(i, L)) + h)

  def rate(dE, beta, gamma):
    # return gamma * np.exp(-0.5 * beta * dE)
    return np.array(gamma * jax.nn.sigmoid(-beta * dE))

  
  K = np.zeros((n_states, n_states))

  # (2^N, N) operations
  for col, sigma in enumerate(states):
    for i in range(N):                          # all spins

      eta    = sigma.copy()
      eta[i] *= -1
      row = state_idx[tuple(eta)]

      dE = dE_flip(sigma, i)
      r = rate(dE, beta[nu], gamma[nu])
      K[row, col] += r

  # print(np.allclose(np.diag(K), 0))
  np.fill_diagonal(K, -K.sum(axis=0))
  return K


In [ ]:
K1 = build_K(0, L, gammas_jax, betas_jax, J, h)
K2 = build_K(1, L, gammas_jax, betas_jax, J, h)
K  = K1 + K2